# 1. ARVO Router 학습

ARVO train split만 사용해 Expert Router를 학습하고 `models/router-arvo.pkl`로 저장합니다. OpenRouter API는 호출하지 않습니다.

In [ ]:
import json
from collections import Counter
from pathlib import Path
from pprint import pprint
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
DATA_DIR = ROOT / 'data' / 'arvo'
MODEL_PATH = ROOT / 'models' / 'router-arvo.pkl'

from llm_security.config import AppConfig
from llm_security.datasets import load_router_samples_jsonl
from llm_security.routing import AdaptiveExpertRouter, RoutingPolicyConfig

## 학습 데이터 확인

In [ ]:
manifest = json.loads((DATA_DIR / 'manifest.json').read_text(encoding='utf-8'))
train_samples = load_router_samples_jsonl(DATA_DIR / 'router_train.jsonl')
family_counts = Counter(
    family.value for sample in train_samples for family in sample.labels
)
print('train projects:', manifest['splits']['train']['project_count'])
print('train samples:', len(train_samples))
print('family distribution:')
pprint(dict(sorted(family_counts.items())))

## Router 학습 및 저장

In [ ]:
config = AppConfig.from_env(ROOT / '.env')
router = AdaptiveExpertRouter.fit(
    train_samples,
    policy_config=RoutingPolicyConfig(
        high_confidence=config.router.high_confidence,
        min_margin=config.router.min_margin,
        max_entropy=config.router.max_entropy,
        max_experts=config.router.max_experts,
    ),
    seed=config.runtime.seed,
    use_rule_fallback=config.router.use_rule_fallback,
)
router.save(MODEL_PATH)
print('training complete')
print('saved model:', MODEL_PATH)
print('learned families:', [family.value for family in router.available_families])